In [1]:
import numpy as np
import json
import random
from collections import deque
from pathlib import Path

In [2]:
GRID_SIZE = 5
NUM_GRIDS = 100

random.seed(42)
np.random.seed(42)

In [3]:
def is_goal_reachable(obstacles, start, goal):
    queue = deque([start])
    visited = {start}

    directions = [
        (-1, 0),  # North
        (1, 0),   # South
        (0, -1),  # West
        (0, 1)    # East
    ]

    while queue:
        r, c = queue.popleft()

        if (r, c) == goal:
            return True

        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            next_pos = (nr, nc)

            if (
                0 <= nr < GRID_SIZE
                and 0 <= nc < GRID_SIZE
                and next_pos not in obstacles
                and next_pos not in visited
            ):
                visited.add(next_pos)
                queue.append(next_pos)

    return False


In [4]:
def generate_grids(num_grids, grid_size):
    grids = []
    seen = set()

    while len(grids) < num_grids:
        start_pos = (random.randint(0, grid_size - 1), random.randint(0, grid_size - 1))
        goal_pos = (random.randint(0, grid_size - 1), random.randint(0, grid_size - 1))
        while goal_pos == start_pos:
            goal_pos = (random.randint(0, grid_size - 1), random.randint(0, grid_size - 1))

        
        open_positions = [
            (r, c)
            for r in range(grid_size)
            for c in range(grid_size)
            if (r, c) not in [start_pos, goal_pos]
        ]


        num_obstacles = random.randint(4, 6)
        obstacles = frozenset(
            random.sample(open_positions, num_obstacles)
        )

        # Unique obstacle configuration
        if obstacles in seen:
            continue

        # Only retain grids with reachable goals
        if is_goal_reachable(obstacles, start_pos, goal_pos):
          seen.add(obstacles)

          grid = np.full((grid_size, grid_size), '.')

          grid[start_pos] = 'S'  # Start
          grid[goal_pos] = 'G'  # Goal

          for r, c in obstacles:
              grid[r, c] = 'B'

          grids.append({
            'id': len(grids),
            'grid': grid.tolist(),
            'start': start_pos,
            'goal': goal_pos,
            'obstacles': list(obstacles)
          })

    return grids

In [5]:
grids = generate_grids(NUM_GRIDS, GRID_SIZE)

print("Number of grids:", len(grids))

Number of grids: 100


In [13]:
for grid in grids:
    print(f"Grid:\n{'\n'.join(' '.join(row) for row in grid['grid'])}")


Grid:
S . . . B
B . . . .
. G . . .
. . . . B
. . . B .
Grid:
. . B . S
. . B B .
. . . . .
G . . B .
. . . . .
Grid:
. . . . .
. . B B .
. . . B B
. . B . .
S G B . .
Grid:
. B . B .
. . . B .
. . G B .
S B . . B
. . . . .
Grid:
. B B . S
. . B . .
. . . . G
. . . . .
B . . . B
Grid:
. G . . .
. . S . .
B . . . B
. B . . .
. . B . .
Grid:
. . . . .
. . . B .
. S G . .
. . . . .
. . B B B
Grid:
. . . . S
. B . B G
B . . . B
. B . . B
. . . . .
Grid:
. G . B .
. . S . .
B . B . B
. . . . .
. . . . .
Grid:
. . . . .
. S . B .
. . B B B
. . . . B
. . . . G
Grid:
. . . B B
B . . . S
. . . . .
G . . . .
. . B . .
Grid:
. . . G .
. . . . B
. . . . .
B . B B S
. B . . .
Grid:
S . . . B
. B . . .
B B . . B
B . . . .
. . G . .
Grid:
. . S . B
B . B . .
B . . . .
. . B . .
. G B . .
Grid:
B . . . .
. . . . .
B S . . .
. B . . .
B . . . G
Grid:
. . S B .
. . . B .
. G . . .
. . B . .
B . . . .
Grid:
. . . . S
. G B . .
B . . . .
B . B B B
. . . . .
Grid:
. . . . .
. . G . S
. . . B .
. B . B .
. 

In [12]:
idx = 10

print(f"Grid:\n{'\n'.join(' '.join(row) for row in grids[idx]['grid'])}")

print("Start position:", grids[idx]['start'])
print("Goal position:", grids[idx]['goal'])
print("Obstacles:", grids[idx]['obstacles'])

Grid:
. . . B B
B . . . S
. . . . .
G . . . .
. . B . .
Start position: (1, 4)
Goal position: (3, 0)
Obstacles: [(1, 0), (0, 3), (0, 4), (4, 2)]


In [11]:
print("Current directory:", Path.cwd())

with open("data/grids.json", "w") as f:
    json.dump(grids, f, indent=2)

Current directory: /home/sm/projects/llm-grid-solver
